# Environment set up

In [13]:
import torch
import math
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cuda


# Load Dataset

In [2]:
dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")

# Train/Validation Split
split_dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

max_train = 15000
max_val = 2000

if len(train_dataset) > max_train:
    train_dataset = train_dataset.shuffle(seed=42).select(range(max_train))
if len(eval_dataset) > max_val:
    eval_dataset = eval_dataset.shuffle(seed=42).select(range(max_val))


README.md: 0.00B [00:00, ?B/s]

Bitext_Sample_Customer_Support_Training_(…):   0%|          | 0.00/19.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

# Load Model and Tokenizer

In [3]:
model_name = "EleutherAI/gpt-neo-125M"
tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(model_name)
base_model.resize_token_embeddings(len(tokenizer))
base_model.to(device)


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/526M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

GPTNeoForCausalLM(
  (transformer): GPTNeoModel(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(2048, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPTNeoBlock(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPTNeoAttention(
          (attention): GPTNeoSelfAttention(
            (attn_dropout): Dropout(p=0.0, inplace=False)
            (resid_dropout): Dropout(p=0.0, inplace=False)
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=False)
            (q_proj): Linear(in_features=768, out_features=768, bias=False)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPTNeoMLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (c_proj): Linear(in_fe

In [4]:
block_size = 256 # UPDATED: Increased sequence length to capture full responses

def tokenize_fn(examples):
    # UPDATED: Added eos_token at the end to teach the model to stop generating
    texts = [
        f"User: {i}\nAssistant: {a}{tokenizer.eos_token}"
        for i, a in zip(examples["instruction"], examples["response"])
    ]
    return tokenizer(texts, truncation=True, max_length=block_size)

# Map the tokenization and clean up columns
tok_train = train_dataset.map(tokenize_fn, batched=True, remove_columns=train_dataset.column_names)
tok_val = eval_dataset.map(tokenize_fn, batched=True, remove_columns=eval_dataset.column_names)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

## LoRA Configuration

In [5]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"]
)

model = get_peft_model(base_model, lora_config)


## Training Arguments & Trainer

In [7]:
output_dir = "gptneo125m-bitext-lora"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=100, 
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    data_collator=data_collator,
)

# Start training
trainer.train()


Epoch,Training Loss,Validation Loss
1,1.471063,1.462768
2,1.325483,1.332423
3,1.280758,1.297817


TrainOutput(global_step=1407, training_loss=1.467281891254135, metrics={'train_runtime': 1897.6362, 'train_samples_per_second': 23.714, 'train_steps_per_second': 0.741, 'total_flos': 5391304869888000.0, 'train_loss': 1.467281891254135, 'epoch': 3.0})

## Save and Merge

In [8]:
adapter_dir = "bitext_lora_adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print("Saved adapter to:", adapter_dir)


Saved adapter to: bitext_lora_adapter


In [9]:
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(model_name).to(device)
merged = PeftModel.from_pretrained(base, adapter_dir)
merged = merged.merge_and_unload()


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
merged_dir = "gptneo125m-bitext-merged"
merged.save_pretrained(merged_dir)
tokenizer.save_pretrained(merged_dir)
print("Merged model saved to:", merged_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to: gptneo125m-bitext-merged


# Testing 

In [15]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load the already trained and merged model from your folder
merged_dir = "gptneo125m-bitext-merged"
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading saved model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(merged_dir)
model = AutoModelForCausalLM.from_pretrained(merged_dir).to(device)

testing_prompts = [
    # 1. Account & Profile
    "I forgot my password. How can I reset it?",
    "How do I change my profile picture?",
    
    # 2. Orders & Payments
    "I placed an order yesterday, but I haven't received a confirmation email.",
    "How can I cancel my order and get a refund?",
    
    # 3. Technical Issues
    "The app keeps crashing when I open it. Any solutions?",
    "Why isn't my promo code working?",
    
    # 4. Product Questions
    "What are the dimensions of product X?",
    "Do you have a warranty or return policy?",
    
    # 5. Friendly / Human Touch Testing
    "I'm really frustrated that my order is late. Can you help?",
]

print("\n--- Running Optimized Inference ---\n")

for prompt in testing_prompts:
    input_text = f"User: {prompt}\nAssistant:"
    inputs = tokenizer(input_text, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,  
            pad_token_id=tokenizer.eos_token_id,
            temperature=0.3,     
            do_sample=True,
            top_p=0.9
        )
    
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    assistant_reply = full_response.split("Assistant:")[-1].strip()
    
    print(f" User: {prompt}")
    print(f" Assistant: {assistant_reply}")
    print("-" * 60)

Loading saved model and tokenizer...


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]


--- Running Optimized Inference ---

 User: I forgot my password. How can I reset it?
 Assistant: I'm on it! I apologize for any inconvenience you have experienced during this process. To reset your password, you can follow these steps:

1. Log in to your account on our website.
2. Navigate to the "My Account" or "Profile" section.
3. Look for the "Password Reset" or "Reset Password" option.
4. Click on it to reset your password.
5. Follow the prompts or instructions provided to complete the reset.
6. Save the changes and proceed with the reset.

If you encounter any difficulties or have any further questions during the process, please don't hesitate to reach out. I'm here to assist you every step of the way. Your satisfaction is our top priority! Enjoy the security and privacy of our users!

If you have any further questions or need further assistance, please don't hesitate to let me know. I'm here to ensure a smooth and secure reset for you!

If you have any further questions or nee

In [16]:
# 9) Push the Merged Model to Hugging Face Hub
from huggingface_hub import notebook_login
from transformers import AutoModelForCausalLM, AutoTokenizer

In [17]:
# 1. Log in to Hugging Face
print("Please paste your Hugging Face WRITE token below:")
notebook_login()

Please paste your Hugging Face WRITE token below:


## publishing

In [18]:
# 2. Define your repo details
# ---> IMPORTANT: Replace with your actual Hugging Face username! <---
hf_username = "nourhan214" 
model_repo_name = f"{hf_username}/gptneo125m-bitext-customer-support"

# 3. Load the saved model and tokenizer from your local directory
merged_dir = "gptneo125m-bitext-merged"
print(f"\nLoading model and tokenizer from {merged_dir}...")
tokenizer = AutoTokenizer.from_pretrained(merged_dir)
model = AutoModelForCausalLM.from_pretrained(merged_dir)

# 4. Push to the Hub
print(f"Pushing model to https://huggingface.co/{model_repo_name} ...")
# This might take a few minutes depending on your internet connection
model.push_to_hub(model_repo_name)
tokenizer.push_to_hub(model_repo_name)

print("\nSuccess! Your model is now live on Hugging Face!")


Loading model and tokenizer from gptneo125m-bitext-merged...


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

Pushing model to https://huggingface.co/nourhan214/gptneo125m-bitext-customer-support ...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]


Success! Your model is now live on Hugging Face!
